In [ ]:
# -----------------------------
# 03_train_pipeline.ipynb — FINAL corrected training pipeline (paste entire cell)
# -----------------------------
import os
import json
import math
import random
import shutil
from pathlib import Path
from time import time
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW   # use torch.optim.AdamW

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, precision_recall_fscore_support

# ----------------- USER CONFIG -----------------
DATA_SPLITS_ROOT = Path("/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/data_splits")
OUTPUT_ROOT_BASE = Path("/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/train_pipeline")

TARGET_LANGS = None  # None => auto-detect languages under DATA_SPLITS_ROOT
MODEL_NAME = "distilbert-base-multilingual-cased"
MAX_EPOCHS = 6
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 50
MAX_GRAD_NORM = 1.0
EARLY_STOPPING_PATIENCE = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
FP16 = False   # set to True only if you wish and runtime supports amp
NUM_WORKERS = 2
SAVE_TOKENIZER = True

# reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ----------------- UTILITIES -----------------
def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)
    return p

def read_label_map(lang_dir: Path) -> Tuple[Dict[str,int], Dict[int,str], Dict]:
    p = lang_dir / "label_map.json"
    if not p.exists():
        raise FileNotFoundError(f"label_map.json not found at {p}. Run 01_dataset_eda to create splits.")
    jd = json.load(open(p, encoding="utf8"))
    label_map = {str(k): int(v) for k,v in jd.get("label_map", {}).items()}
    inv_label_map = {int(v): str(k) for k,v in jd.get("label_map", {}).items()}
    metadata = jd.get("metadata", {})
    return label_map, inv_label_map, metadata

def load_dataframe(lang_dir: Path, split_name: str) -> pd.DataFrame:
    p = lang_dir / f"{split_name}.csv"
    if not p.exists():
        raise FileNotFoundError(f"Expected split at {p}")
    df = pd.read_csv(p)
    if 'label' not in df.columns and 'label_id' not in df.columns:
        raise RuntimeError(f"CSV {p} must contain 'label' or 'label_id' column.")
    if 'text' not in df.columns and 'file_path' not in df.columns:
        raise RuntimeError(f"CSV {p} must contain 'file_path' or 'text' column so we can read script content.")
    return df

def read_text_from_row(row) -> str:
    if 'text' in row and not pd.isna(row['text']):
        return str(row['text'])
    if 'file_path' in row and not pd.isna(row['file_path']):
        fp = Path(row['file_path'])
        if not fp.exists():
            try:
                return Path(str(row['file_path'])).read_text(encoding="utf8", errors="ignore")
            except Exception:
                return ""
        else:
            return fp.read_text(encoding="utf8", errors="ignore")
    return ""

# ----------------- CHUNKING (fixed: return_tensors=None, convert each chunk manually) -----------------
def chunk_text_with_tokenizer(tokenizer, text: str, max_len: int, stride: int) -> List[Dict[str, torch.Tensor]]:
    """
    Tokenize and produce chunks using return_overflowing_tokens with truncation=True.
    Use return_tensors=None to receive lists; convert each chunk to a torch tensor individually.
    """
    if text is None:
        text = ""
    enc = tokenizer(
        text,
        truncation=True,
        return_overflowing_tokens=True,
        return_offsets_mapping=False,
        max_length=max_len,
        stride=stride,
        padding=False,
        return_tensors=None
    )

    chunks = []
    # tokenizer returns lists of lists when return_tensors=None
    input_ids_list = enc.get("input_ids", [])
    attention_mask_list = enc.get("attention_mask", None)
    for i, ids in enumerate(input_ids_list):
        mask = attention_mask_list[i] if attention_mask_list is not None else [1]*len(ids)
        input_ids_tensor = torch.tensor(ids, dtype=torch.long).unsqueeze(0)
        attention_mask_tensor = torch.tensor(mask, dtype=torch.long).unsqueeze(0)
        chunks.append({
            "input_ids": input_ids_tensor,
            "attention_mask": attention_mask_tensor
        })
    return chunks

# Flatten per-file chunks into dataset items
class ChunkDataset(Dataset):
    def __init__(self, file_entries: List[Dict]):
        self.samples = []
        for file_idx, fe in enumerate(file_entries):
            for chunk in fe['chunks']:
                self.samples.append({
                    "input_ids": chunk['input_ids'].squeeze(0),
                    "attention_mask": chunk['attention_mask'].squeeze(0),
                    "label_id": int(fe['label_id']),
                    "file_path": fe['file_path'],
                    "file_idx": file_idx
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        return s

def collate_fn(batch):
    input_ids = [b['input_ids'] for b in batch]
    attention_mask = [b['attention_mask'] for b in batch]
    labels = torch.tensor([b['label_id'] for b in batch], dtype=torch.long)
    file_paths = [b['file_path'] for b in batch]
    file_idxs = [b['file_idx'] for b in batch]
    input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=0)
    attention_mask = torch.nn.utils.rnn.pad_sequence(attention_mask, batch_first=True, padding_value=0)
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "file_paths": file_paths,
        "file_idxs": file_idxs
    }

def aggregate_chunk_logits_to_file(pred_logits_per_chunk_files: Dict[str, np.ndarray]) -> Dict[str, np.ndarray]:
    agg = {}
    for fp, logits_arr in pred_logits_per_chunk_files.items():
        if logits_arr.shape[0] == 0:
            agg[fp] = np.zeros((logits_arr.shape[1],), dtype=float)
        else:
            agg[fp] = logits_arr.mean(axis=0)
    return agg

def compute_file_level_predictions_from_logits(logits_map: Dict[str, np.ndarray], inv_label_map: Dict[int,str]) -> pd.DataFrame:
    rows = []
    for fp, logits in logits_map.items():
        if logits is None or logits.size == 0:
            pred_id = -1
            pred_label = None
        else:
            pred_id = int(np.argmax(logits))
            pred_label = inv_label_map.get(pred_id, str(pred_id))
        rows.append({"file_path": fp, "pred_label_id": pred_id, "pred_label_str": pred_label})
    return pd.DataFrame(rows)

def compute_classification_metrics(y_true: List[int], y_pred: List[int], inv_label_map: Dict[int,str]):
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    cls_report = classification_report(y_true, y_pred, target_names=[inv_label_map[i] for i in sorted(inv_label_map.keys())], zero_division=0, output_dict=True)
    cm = confusion_matrix(y_true, y_pred).tolist()
    return {"accuracy": acc, "macro_f1": macro_f1, "classification_report": cls_report, "confusion_matrix": cm}

# ----------------- MAIN LOOP (per language) -----------------
if TARGET_LANGS is None:
    TARGET_LANGS = sorted([d.name for d in DATA_SPLITS_ROOT.iterdir() if d.is_dir()])

print(f"[INFO] Languages to process: {TARGET_LANGS}")
for LANG in TARGET_LANGS:
    print("\n" + "="*80)
    print(f"[LANGUAGE] {LANG}")
    lang_dir = DATA_SPLITS_ROOT / LANG
    if not lang_dir.exists():
        print(f"[WARN] Language split folder not found: {lang_dir}. Skipping.")
        continue

    EXP_ROOT = ensure_dir(OUTPUT_ROOT_BASE / LANG)
    EXP_OUT = ensure_dir(EXP_ROOT / ("teacher_" + MODEL_NAME.replace("/", "_")))
    LOGITS_DIR = ensure_dir(EXP_OUT / "logits")
    MODEL_DIR = ensure_dir(EXP_OUT / "best_model")
    ensure_dir(EXP_OUT / "checkpoints")

    label_map, inv_label_map, metadata = read_label_map(lang_dir)
    num_labels = len(label_map)
    print(f"[INFO] label_map: {label_map}  (num_labels={num_labels})")
    CHUNK_MAX_LEN = int(metadata.get("chunk_max_len", 256))
    CHUNK_STRIDE = int(metadata.get("chunk_stride", 64))
    print(f"[INFO] chunk params max_len={CHUNK_MAX_LEN}, stride={CHUNK_STRIDE}")

    train_df = load_dataframe(lang_dir, "train")
    val_df   = load_dataframe(lang_dir, "val")
    test_df  = load_dataframe(lang_dir, "test")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=num_labels)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config)
    model.to(DEVICE)

    def prepare_file_entries(df: pd.DataFrame):
        entries = []
        for idx, row in tqdm(df.iterrows(), total=len(df), desc="Chunking files"):
            file_path = str(row.get("file_path", "")) if 'file_path' in row else None
            text = read_text_from_row(row)
            if text is None:
                text = ""
            chunks = chunk_text_with_tokenizer(tokenizer, text, max_len=CHUNK_MAX_LEN, stride=CHUNK_STRIDE)
            if 'label_id' in row and not pd.isna(row['label_id']):
                label_id = int(row['label_id'])
            else:
                label_id = int(label_map.get(str(row['label']), -1))
            entries.append({
                "file_path": str(file_path) if file_path else f"{LANG}_generated_{idx}",
                "label_id": label_id,
                "chunks": chunks
            })
        return entries

    print("[INFO] Preparing train file entries (chunked)...")
    train_entries = prepare_file_entries(train_df)
    print("[INFO] Preparing val file entries (chunked)...")
    val_entries = prepare_file_entries(val_df)
    print("[INFO] Preparing test file entries (chunked)...")
    test_entries = prepare_file_entries(test_df)

    train_dataset = ChunkDataset(train_entries)
    val_dataset = ChunkDataset(val_entries)
    test_dataset = ChunkDataset(test_entries)

    train_loader = DataLoader(train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True,
                              collate_fn=collate_fn, num_workers=NUM_WORKERS)
    val_loader = DataLoader(val_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False,
                            collate_fn=collate_fn, num_workers=NUM_WORKERS)
    test_loader = DataLoader(test_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False,
                             collate_fn=collate_fn, num_workers=NUM_WORKERS)

    no_decay = ["bias", "LayerNorm.weight"]
    optimizer_grouped_parameters = [
        {
            "params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
            "weight_decay": WEIGHT_DECAY,
        },
        {
            "params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
            "weight_decay": 0.0,
        },
    ]
    optimizer = AdamW(optimizer_grouped_parameters, lr=LEARNING_RATE, eps=1e-8)
    total_steps = len(train_loader) * MAX_EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=WARMUP_STEPS, num_training_steps=total_steps)

    best_val_macro_f1 = -1.0
    best_epoch = -1
    epochs_no_improve = 0
    global_step = 0
    history = {"train_loss": [], "val_loss": [], "val_macro_f1": []}

    scaler = torch.cuda.amp.GradScaler() if (FP16 and torch.cuda.is_available()) else None

    print("[TRAINING] Starting training loop on device:", DEVICE)
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        step = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch} train", leave=False)
        for batch in pbar:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)

            optimizer.zero_grad()
            if scaler:
                with torch.cuda.amp.autocast():
                    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                    loss = outputs.loss
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                loss = outputs.loss
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                optimizer.step()
            scheduler.step()
            total_loss += loss.item()
            step += 1
            global_step += 1
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})
        avg_train_loss = total_loss / max(1, step)
        history["train_loss"].append(avg_train_loss)

        model.eval()
        val_logits_by_file = {fe['file_path']: [] for fe in val_entries}
        val_loss = 0.0
        val_steps = 0

        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch} val", leave=False):
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                labels = batch['labels'].to(DEVICE)
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits.detach().cpu().numpy()
                for i, fp in enumerate(batch['file_paths']):
                    val_logits_by_file[fp].append(logits[i])
                loss_fct = nn.CrossEntropyLoss()
                loss_val = loss_fct(torch.tensor(logits), labels.detach().cpu()).item()
                val_loss += loss_val
                val_steps += 1

        for fp in val_logits_by_file:
            if val_logits_by_file[fp]:
                val_logits_by_file[fp] = np.vstack(val_logits_by_file[fp])
            else:
                val_logits_by_file[fp] = np.zeros((0, num_labels), dtype=float)
        val_agg_logits = aggregate_chunk_logits_to_file(val_logits_by_file)

        y_true = []
        y_pred = []
        for fe in val_entries:
            fp = fe['file_path']
            true_id = int(fe['label_id'])
            logits = val_agg_logits.get(fp, np.zeros((num_labels,), dtype=float))
            if logits.size == 0:
                pred_id = -1
            else:
                pred_id = int(np.argmax(logits))
            y_true.append(true_id)
            y_pred.append(pred_id)

        y_pred_for_metrics = [p if p >= 0 else 0 for p in y_pred]
        val_metrics = compute_classification_metrics(y_true, y_pred_for_metrics, inv_label_map)
        avg_val_loss = (val_loss / max(1, val_steps)) if val_steps else 0.0
        history["val_loss"].append(avg_val_loss)
        history["val_macro_f1"].append(val_metrics["macro_f1"])

        print(f"[EPOCH {epoch}] train_loss={avg_train_loss:.4f} val_loss={avg_val_loss:.4f} val_macro_f1={val_metrics['macro_f1']:.4f}")

        current_val_macro_f1 = val_metrics["macro_f1"]
        if current_val_macro_f1 > best_val_macro_f1 + 1e-6:
            print(f"[MODEL] New best val_macro_f1: {current_val_macro_f1:.4f} (improved from {best_val_macro_f1:.4f}) -> saving checkpoint")
            best_val_macro_f1 = current_val_macro_f1
            best_epoch = epoch
            epochs_no_improve = 0
            model.save_pretrained(MODEL_DIR)
            if SAVE_TOKENIZER:
                tokenizer.save_pretrained(MODEL_DIR)
            json.dump({"label_map": label_map, "inv_label_map": {str(k): v for k,v in inv_label_map.items()}, "metadata": metadata},
                      open(MODEL_DIR / "label_map.json","w",encoding="utf8"), indent=2, ensure_ascii=False)
            json.dump({"epoch": epoch, "val_metrics": val_metrics, "train_loss": avg_train_loss}, open(MODEL_DIR / "best_epoch_metrics.json", "w", encoding="utf8"), indent=2, ensure_ascii=False)
        else:
            epochs_no_improve += 1
            print(f"[EARLY_STOPPING] No improvement for {epochs_no_improve} epochs (patience={EARLY_STOPPING_PATIENCE})")

        if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
            print(f"[EARLY_STOPPING] Stopping early at epoch {epoch}")
            break

    if not (MODEL_DIR.exists() and any(MODEL_DIR.iterdir())):
        print("[WARN] Best model was not saved during training. Saving last model as best.")
        model.save_pretrained(MODEL_DIR)
        tokenizer.save_pretrained(MODEL_DIR)
        json.dump({"label_map": label_map, "inv_label_map": {str(k): v for k,v in inv_label_map.items()}, "metadata": metadata},
                  open(MODEL_DIR / "label_map.json","w",encoding="utf8"), indent=2, ensure_ascii=False)

    # ----------------- Generate and save teacher chunk logits for train & val -----------------
    print("[LOGITS] Generating teacher chunk logits (train & val) for distillation notebook")
    teacher_tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
    teacher_model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR))
    teacher_model.to(DEVICE)
    teacher_model.eval()

    def compute_chunk_logits_for_entries(entries, out_map):
        with torch.no_grad():
            items = []
            for fe in entries:
                for ch in fe['chunks']:
                    items.append((fe['file_path'], ch['input_ids'].squeeze(0), ch['attention_mask'].squeeze(0)))
            bs = EVAL_BATCH_SIZE
            all_by_file = {}
            for i in range(0, len(items), bs):
                batch_items = items[i:i+bs]
                input_ids = torch.nn.utils.rnn.pad_sequence([it[1] for it in batch_items], batch_first=True, padding_value=0).to(DEVICE)
                attention_mask = torch.nn.utils.rnn.pad_sequence([it[2] for it in batch_items], batch_first=True, padding_value=0).to(DEVICE)
                out = teacher_model(input_ids=input_ids, attention_mask=attention_mask)
                logits = out.logits.detach().cpu().numpy()
                for j, (fp, _, _) in enumerate(batch_items):
                    if fp not in all_by_file:
                        all_by_file[fp] = []
                    all_by_file[fp].append(logits[j])
            for fp, arr in all_by_file.items():
                out_map[fp] = np.vstack(arr) if arr else np.zeros((0, num_labels), dtype=float)
        return out_map

    teacher_train_map = {}
    teacher_val_map = {}
    teacher_train_map = compute_chunk_logits_for_entries(train_entries, teacher_train_map)
    teacher_val_map = compute_chunk_logits_for_entries(val_entries, teacher_val_map)

    np_save_train = LOGITS_DIR / "teacher_train_chunk_logits.npz"
    np_save_val = LOGITS_DIR / "teacher_val_chunk_logits.npz"
    np.savez_compressed(str(np_save_train), **teacher_train_map)
    np.savez_compressed(str(np_save_val), **teacher_val_map)
    print(f"[LOGITS] Saved teacher chunk logits: {np_save_train} (train), {np_save_val} (val)")

    # ----------------- Evaluate on Test (file-level predictions) -----------------
    print("[EVAL] Running file-level test evaluation using best model")
    model_eval = AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR)).to(DEVICE)
    test_logits_map = {}
    test_logits_map = compute_chunk_logits_for_entries(test_entries, test_logits_map)
    test_agg_logits = aggregate_chunk_logits_to_file(test_logits_map)

    test_rows = []
    y_true = []
    y_pred = []
    for fe in test_entries:
        fp = fe['file_path']
        true_id = int(fe['label_id'])
        logits = test_agg_logits.get(fp, np.zeros((num_labels,), dtype=float))
        if logits.size == 0:
            pred_id = -1
            pred_label = None
        else:
            pred_id = int(np.argmax(logits))
            pred_label = inv_label_map.get(pred_id, str(pred_id))
        test_rows.append({
            "file_path": fp,
            "gold_label_id": true_id,
            "gold_label_str": inv_label_map.get(true_id, str(true_id)),
            "pred_label_id": pred_id,
            "pred_label_str": pred_label
        })
        y_true.append(true_id)
        y_pred.append(pred_id if pred_id >= 0 else 0)

    preds_df = pd.DataFrame(test_rows)
    preds_csv_path = EXP_OUT / "predictions.csv"
    preds_df.to_csv(preds_csv_path, index=False, encoding="utf8")
    print(f"[EVAL] Saved predictions CSV: {preds_csv_path}")

    test_metrics = compute_classification_metrics(y_true, y_pred, inv_label_map)
    metrics_path = EXP_OUT / "metrics.json"
    json.dump(test_metrics, open(metrics_path, "w", encoding="utf8"), indent=2, ensure_ascii=False)
    print(f"[EVAL] Saved metrics JSON: {metrics_path}")

    summary = {
        "language": LANG,
        "model_name": MODEL_NAME,
        "num_labels": num_labels,
        "label_map": label_map,
        "inv_label_map": inv_label_map,
        "metadata": metadata,
        "training": {
            "max_epochs": MAX_EPOCHS,
            "train_batch_size": TRAIN_BATCH_SIZE,
            "eval_batch_size": EVAL_BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "best_epoch": best_epoch,
            "best_val_macro_f1": best_val_macro_f1,
            "history": history
        },
        "paths": {
            "exp_out": str(EXP_OUT.resolve()),
            "model_dir": str(MODEL_DIR.resolve()),
            "predictions_csv": str(preds_csv_path.resolve()),
            "metrics_json": str(metrics_path.resolve()),
            "logits_dir": str(LOGITS_DIR.resolve())
        }
    }
    summary_path = EXP_OUT / "summary.json"
    json.dump(summary, open(summary_path, "w", encoding="utf8"), indent=2, ensure_ascii=False)
    print(f"[DONE] Saved summary: {summary_path}")
    print("="*80)

print("\n[ALL DONE] Training pipeline completed for all requested languages.")


[INFO] Languages to process: ['English', 'Hindi', 'Marathi']

[LANGUAGE] English
[INFO] label_map: {'G': 0, 'PG': 1, 'PG-13': 2, 'R': 3, 'NC-17': 4}  (num_labels=5)
[INFO] chunk params max_len=256, stride=64


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

...

[TRAINING] Starting training loop on device: cuda


[EPOCH 1] train_loss=0.6527 val_loss=0.8340 val_macro_f1=0.3891
[MODEL] New best val_macro_f1: 0.3891 (improved from -1.0000) -> saving checkpoint


[EPOCH 2] train_loss=0.5596 val_loss=0.9533 val_macro_f1=0.5982
[MODEL] New best val_macro_f1: 0.5982 (improved from 0.3891) -> saving checkpoint


[EPOCH 3] train_loss=0.4285 val_loss=1.1521 val_macro_f1=0.5833
[EARLY_STOPPING] No improvement for 1 epochs (patience=2)


[EPOCH 4] train_loss=0.3385 val_loss=1.7363 val_macro_f1=0.4976
[EARLY_STOPPING] No improvement for 2 epochs (patience=2)
[EARLY_STOPPING] Stopping early at epoch 4
[LOGITS] Generating teacher chunk logits (train & val) for distillation notebook
[LOGITS] Saved teacher chunk logits: /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/train_pipeline/Marathi/teacher_distilbert-base-multilingual-cased/logits/teacher_train_chunk_logits.npz (train), /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/train_pipeline/Marathi/teacher_distilbert-base-multilingual-cased/logits/teacher_val_chunk_logits.npz (val)
[EVAL] Running file-level test evaluation using best model
[EVAL] Saved predictions CSV: /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/train_pipeline/Marathi/teacher_distilbert-base-multilingual-cased/predictions.csv
[EVAL] Saved metrics JSON: /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/

In [ ]:
# -----------------------------
# Sanity-check cell to run AFTER 03_train_pipeline and BEFORE 04_distillation
# Paste entire cell and run.
# -----------------------------
import json, os
from pathlib import Path
from collections import defaultdict

# ---------- CONFIG ----------
DATA_SPLITS_ROOT = Path("/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/data_splits")
TRAIN_PIPELINE_OUTPUT_ROOT = Path("/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/train_pipeline")
AUTO_FIX = False   # set True to create a non-destructive mapping file when basename matches are found
SAMPLE_MAX = 10    # how many examples to print for each type of mismatch
# -------------------------

def find_teacher_exp_dir(lang: str):
    lang_root = TRAIN_PIPELINE_OUTPUT_ROOT / lang
    if not lang_root.exists():
        return None
    candidates = [d for d in lang_root.iterdir() if d.is_dir() and d.name.startswith("teacher_")]
    if not candidates:
        # fallback: any dir with 'best_model'
        candidates = [d for d in lang_root.iterdir() if d.is_dir() and (d / "best_model").exists()]
    if not candidates:
        return None
    return sorted(candidates)[-1]

def safe_load_npz_keys(npz_path: Path):
    try:
        import numpy as np
        npz = np.load(str(npz_path), allow_pickle=True)
        keys = list(npz.files)
        return keys
    except Exception as e:
        print(f"[ERR] Could not load {npz_path}: {e}")
        return []

def collect_csv_filepaths(lang_dir: Path):
    files = {}
    for split in ["train","val","test"]:
        p = lang_dir / f"{split}.csv"
        if not p.exists():
            files[split] = None
            continue
        import pandas as pd
        df = pd.read_csv(p)
        if 'file_path' in df.columns:
            fps = [str(x) for x in df['file_path'].tolist()]
        elif 'text' in df.columns:
            fps = [None] * len(df)
        else:
            fps = [None] * len(df)
        files[split] = fps
    return files

def check_labelmap_consistency(lang_dir: Path):
    jm = lang_dir / "label_map.json"
    out = {}
    if not jm.exists():
        out['exists'] = False
        return out
    out['exists'] = True
    jd = json.load(open(jm,encoding="utf8"))
    label_map = jd.get("label_map", {})
    inv_label_map = {int(v):k for k,v in label_map.items()}
    metadata = jd.get("metadata", {})
    out['label_map'] = label_map
    out['inv_label_map'] = inv_label_map
    out['metadata'] = metadata
    # Quick check: ensure label ids are 0..N-1 contiguous
    if label_map:
        ids = sorted([int(v) for v in label_map.values()])
        out['id_sequence_ok'] = (ids == list(range(len(ids))))
    else:
        out['id_sequence_ok'] = False
    # Check csv label_id columns consistency
    from glob import glob
    import pandas as pd
    inconsistencies = []
    for split in ["train","val","test"]:
        p = lang_dir / f"{split}.csv"
        if not p.exists(): continue
        df = pd.read_csv(p)
        if 'label_id' in df.columns:
            bad = df[~df['label_id'].isin(ids)]
            if len(bad) > 0:
                inconsistencies.append({"split": split, "count_bad": len(bad), "sample_bad": bad.head(SAMPLE_MAX).to_dict(orient='records')})
    out['label_id_inconsistencies'] = inconsistencies
    return out

# -- main loop over languages --
langs = sorted([d.name for d in DATA_SPLITS_ROOT.iterdir() if d.is_dir()])
if not langs:
    raise RuntimeError(f"No language split directories found under {DATA_SPLITS_ROOT}. Run 01_dataset_eda.ipynb first.")

report = {}
for lang in langs:
    print("\n" + "="*80)
    print(f"[LANG] {lang}")
    lang_dir = DATA_SPLITS_ROOT / lang
    lang_report = {}
    # 1) Check label_map
    lm = check_labelmap_consistency(lang_dir)
    lang_report['label_map'] = lm
    if not lm['exists']:
        print(f"[ERR] Missing label_map.json in {lang_dir}. Run 01_dataset_eda/03_train_pipeline to create it.")
        report[lang] = lang_report
        continue
    else:
        print(f"[OK] Found label_map.json. labels = {list(lm['label_map'].keys())}; metadata={lm['metadata']}")
        if not lm['id_sequence_ok']:
            print(f"[WARN] label_map ids are not contiguous 0..N-1: {lm['label_map']}")
        if lm['label_id_inconsistencies']:
            print(f"[WARN] Found label_id inconsistencies in CSVs:")
            for inc in lm['label_id_inconsistencies']:
                print(f"  - split={inc['split']} bad_count={inc['count_bad']} sample_bad={inc['sample_bad'][:SAMPLE_MAX]}")
    # 2) Check presence of splits & collect file_path values
    csv_files = {}
    csv_fps = []
    for split in ['train','val','test']:
        p = lang_dir / f"{split}.csv"
        if not p.exists():
            print(f"[ERR] Missing split {p}")
            csv_files[split] = None
            continue
        import pandas as pd
        df = pd.read_csv(p)
        # ensure columns
        has_fp = 'file_path' in df.columns
        has_text = 'text' in df.columns
        has_label = 'label' in df.columns or 'label_id' in df.columns
        print(f"[OK] {split}.csv: rows={len(df)} columns={list(df.columns)}")
        if not has_fp and not has_text:
            print(f"[WARN] {p} has neither 'file_path' nor 'text' column. Downstream code expects 'file_path' to match teacher logits keys.")
        if not has_label:
            print(f"[ERR] {p} missing 'label' or 'label_id' -> required.")
        fps = list(df['file_path'].astype(str).tolist()) if 'file_path' in df.columns else [None]*len(df)
        csv_files[split] = p
        csv_fps.extend([s for s in fps if s is not None])
    lang_report['csv_files'] = {k: str(v) if v else None for k,v in csv_files.items()}
    # 3) Find teacher experiment & logits npz
    teacher_dir = None
    # try to find teacher dir
    training_lang_root = TRAIN_PIPELINE_OUTPUT_ROOT / lang
    if training_lang_root.exists():
        # prefer folder starting with teacher_
        cand = sorted([d for d in training_lang_root.iterdir() if d.is_dir() and d.name.startswith("teacher_")])
        if not cand:
            cand = sorted([d for d in training_lang_root.iterdir() if d.is_dir() and (d/"best_model").exists()])
        if cand:
            teacher_dir = cand[-1]
    if not teacher_dir:
        print(f"[ERR] Could not find teacher experiment dir under {training_lang_root}. You must run 03_train_pipeline first.")
        lang_report['teacher_dir'] = None
        report[lang] = lang_report
        continue
    print(f"[OK] Teacher experiment dir: {teacher_dir}")
    lang_report['teacher_dir'] = str(teacher_dir)
    logits_dir = teacher_dir / "logits"
    train_logits_npz = logits_dir / "teacher_train_chunk_logits.npz"
    val_logits_npz = logits_dir / "teacher_val_chunk_logits.npz"
    if not train_logits_npz.exists() or not val_logits_npz.exists():
        print(f"[ERR] Expected teacher logits not found: {train_logits_npz} or {val_logits_npz}")
        lang_report['logits_exist'] = False
        report[lang] = lang_report
        continue
    lang_report['logits_exist'] = True
    # 4) load npz keys
    train_keys = safe_load_npz_keys(train_logits_npz)
    val_keys = safe_load_npz_keys(val_logits_npz)
    lang_report['teacher_train_keys_count'] = len(train_keys)
    lang_report['teacher_val_keys_count'] = len(val_keys)
    print(f"[OK] Loaded teacher_train keys={len(train_keys)} teacher_val keys={len(val_keys)}")
    # 5) Compare CSV file_path values to teacher npz keys
    csv_fp_set = set(csv_fps)
    npz_key_set = set(train_keys) | set(val_keys)
    exact_matches = sorted(list(csv_fp_set & npz_key_set))
    missing_in_npz = sorted(list(csv_fp_set - npz_key_set))
    extra_in_npz = sorted(list(npz_key_set - csv_fp_set))
    print(f"Exact matches between CSV file_path and teacher npz keys: {len(exact_matches)}")
    print(f"CSV file_path entries missing in teacher npz: {len(missing_in_npz)}")
    print(f"NPZ keys not present in CSV file_path: {len(extra_in_npz)}")
    if exact_matches:
        print("  sample exact match keys:", exact_matches[:min(SAMPLE_MAX, len(exact_matches))])
    if missing_in_npz:
        print("  sample missing_in_npz:", missing_in_npz[:min(SAMPLE_MAX, len(missing_in_npz))])
    if extra_in_npz:
        print("  sample extra_in_npz:", extra_in_npz[:min(SAMPLE_MAX, len(extra_in_npz))])

    # 6) try basename matching to rescue common absolute/relative mismatches
    csv_basename_map = defaultdict(list)
    for fp in csv_fp_set:
        csv_basename_map[Path(fp).name].append(fp)
    npz_basename_map = defaultdict(list)
    for k in npz_key_set:
        npz_basename_map[Path(k).name].append(k)

    basename_matches = []
    basename_only_missing = []
    for bname, csv_list in csv_basename_map.items():
        npz_list = npz_basename_map.get(bname, [])
        if npz_list:
            # create mapping entries
            for csv_fp in csv_list:
                # if multiple npz keys for same basename, keep all
                for npz_k in npz_list:
                    basename_matches.append((csv_fp, npz_k))
        else:
            basename_only_missing.append(bname)

    print(f"Basename matches found: {len(basename_matches)} (csv_fp -> npz_key pairs)")
    if basename_matches:
        print("  sample basename matches:", basename_matches[:min(SAMPLE_MAX, len(basename_matches))])
    if basename_only_missing:
        print(f"Basenames present in CSV but no matching NPZ key found for these basenames (sample): {basename_only_missing[:min(SAMPLE_MAX, len(basename_only_missing))]}")

    lang_report['exact_matches_count'] = len(exact_matches)
    lang_report['missing_in_npz_count'] = len(missing_in_npz)
    lang_report['extra_in_npz_count'] = len(extra_in_npz)
    lang_report['basename_matches_count'] = len(basename_matches)

    # 7) Optionally write a non-destructive mapping file to help downstream notebooks align keys
    mapping_path = logits_dir / "file_path_key_mapping.json"
    if AUTO_FIX and basename_matches:
        # Build mapping: prefer exact matches; else map first npz key for that basename to csv path(s)
        mapping = {}
        # exact matches first
        for k in exact_matches:
            mapping[k] = k
        # use basename matches for missing ones
        for csv_fp, npz_k in basename_matches:
            if csv_fp not in mapping:
                mapping[csv_fp] = npz_k
        # write mapping
        json.dump(mapping, open(mapping_path, "w", encoding="utf8"), indent=2, ensure_ascii=False)
        print(f"[AUTO_FIX] wrote mapping file to {mapping_path} (non-destructive). Downstream notebooks can use this to align keys.")
        lang_report['mapping_written'] = str(mapping_path)
    else:
        if AUTO_FIX:
            print(f"[AUTO_FIX] No basename matches found; no mapping created.")
        else:
            print(f"[INFO] AUTO_FIX disabled. If you want the notebook to produce a mapping file (non-destructive), re-run with AUTO_FIX=True.")

    report[lang] = lang_report

# Save overall report
OUT_REPORT = Path("sanity_check_report.json")
json.dump(report, open(OUT_REPORT, "w", encoding="utf8"), indent=2, ensure_ascii=False)
print("\nSanity-check completed. Full report written to:", OUT_REPORT.resolve())
print("If there are missing entries in teacher logits for CSV files, re-run teacher logits generation in 03_train_pipeline or set AUTO_FIX=True to create a mapping file (basename-based) to help distillation match keys.")



[LANG] English
[OK] Found label_map.json. labels = ['G', 'PG', 'PG-13', 'R', 'NC-17']; metadata={'chunk_max_len': 256, 'chunk_stride': 64}
[OK] train.csv: rows=799 columns=['file_path', 'filename', 'label', 'title', 'year', 'language']
[OK] val.csv: rows=171 columns=['file_path', 'filename', 'label', 'title', 'year', 'language']
[OK] test.csv: rows=172 columns=['file_path', 'filename', 'label', 'title', 'year', 'language']
[OK] Teacher experiment dir: /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/train_pipeline/English/teacher_distilbert-base-multilingual-cased
[OK] Loaded teacher_train keys=799 teacher_val keys=171
Exact matches between CSV file_path and teacher npz keys: 970
CSV file_path entries missing in teacher npz: 172
NPZ keys not present in CSV file_path: 0
  sample exact match keys: ['/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/dataset/English/G_101 Dalmatians_1996.txt', '/content/drive/MyDrive/PhDWorks/4_Final_

In [ ]:
# Create non-destructive file_path -> npz-key mapping (AUTO_FIX)
import json
from pathlib import Path
from collections import defaultdict

DATA_SPLITS_ROOT = Path("/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/data_splits")
TRAIN_PIPELINE_OUTPUT_ROOT = Path("/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/train_pipeline")

langs = sorted([d.name for d in DATA_SPLITS_ROOT.iterdir() if d.is_dir()])
for LANG in langs:
    lang_dir = DATA_SPLITS_ROOT / LANG
    # find teacher exp
    train_lang_root = TRAIN_PIPELINE_OUTPUT_ROOT / LANG
    candidates = [d for d in train_lang_root.iterdir() if d.is_dir() and d.name.startswith("teacher_")] if train_lang_root.exists() else []
    if not candidates:
        candidates = [d for d in train_lang_root.iterdir() if d.is_dir() and (d/"best_model").exists()] if train_lang_root.exists() else []
    if not candidates:
        print(f"[SKIP] No teacher experiment found for {LANG} at {train_lang_root}")
        continue
    teacher_exp_dir = sorted(candidates)[-1]
    logits_dir = teacher_exp_dir / "logits"
    train_npz = logits_dir / "teacher_train_chunk_logits.npz"
    val_npz = logits_dir / "teacher_val_chunk_logits.npz"
    if not train_npz.exists() and not val_npz.exists():
        print(f"[SKIP] No teacher npz logits for {LANG} at {logits_dir}")
        continue
    # load keys
    import numpy as np
    keys = set()
    if train_npz.exists():
        keys |= set(np.load(str(train_npz), allow_pickle=True).files)
    if val_npz.exists():
        keys |= set(np.load(str(val_npz), allow_pickle=True).files)
    # load all CSV file_path entries
    import pandas as pd
    csv_paths = []
    for split in ["train","val","test"]:
        p = lang_dir / f"{split}.csv"
        if p.exists():
            df = pd.read_csv(p)
            if 'file_path' in df.columns:
                csv_paths.extend([str(x) for x in df['file_path'].tolist()])
    csv_set = set(csv_paths)
    # build basename maps
    npz_basename_map = defaultdict(list)
    for k in keys:
        npz_basename_map[Path(k).name].append(k)
    csv_basename_map = defaultdict(list)
    for c in csv_set:
        csv_basename_map[Path(c).name].append(c)
    # Build mapping: prefer exact matches then basename-first mapping
    mapping = {}
    exact = csv_set & keys
    for k in exact:
        mapping[k] = k
    # for csv paths not exact, try basename mapping
    for bname, csv_list in csv_basename_map.items():
        npz_list = npz_basename_map.get(bname, [])
        if not npz_list:
            continue
        for csv_fp in csv_list:
            if csv_fp not in mapping:
                # if multiple candidate npz keys exist, choose the first (non-destructive)
                mapping[csv_fp] = npz_list[0]
    # write mapping only if it contains basename matches
    if mapping:
        out_path = logits_dir / "file_path_key_mapping.json"
        json.dump(mapping, open(out_path, "w", encoding="utf8"), indent=2, ensure_ascii=False)
        print(f"[OK] Wrote mapping for {LANG} -> {out_path} (mapped {len(mapping)} CSV paths).")
    else:
        print(f"[NO MAP] No basename-based mapping possible for {LANG}. Keys={len(keys)} CSV_paths={len(csv_set)}")


[OK] Wrote mapping for English -> /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/train_pipeline/English/teacher_distilbert-base-multilingual-cased/logits/file_path_key_mapping.json (mapped 970 CSV paths).
[OK] Wrote mapping for Hindi -> /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/train_pipeline/Hindi/teacher_distilbert-base-multilingual-cased/logits/file_path_key_mapping.json (mapped 172 CSV paths).
[OK] Wrote mapping for Marathi -> /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/train_pipeline/Marathi/teacher_distilbert-base-multilingual-cased/logits/file_path_key_mapping.json (mapped 84 CSV paths).


In [3]:
import numpy as np
import json
from pathlib import Path
from sklearn.metrics import brier_score_loss
import torch
from torch.utils.data import DataLoader

# Assuming these variables are defined in the previous cells
# DATA_SPLITS_ROOT
# TRAIN_PIPELINE_OUTPUT_ROOT
# collate_fn
# ChunkDataset
# compute_chunk_logits_for_entries
# read_text_from_row
# chunk_text_with_tokenizer
# read_label_map
# load_dataframe

# Function to calculate ECE (using equal width binning)
def expected_calibration_error(y_true, y_probs, n_bins=10):
    """
    Calculates the Expected Calibration Error.

    Args:
        y_true (np.ndarray): True labels (one-hot encoded or integer).
        y_probs (np.ndarray): Predicted probabilities for each class.
        n_bins (int): Number of bins for calibration.

    Returns:
        float: ECE score.
    """
    if y_probs.ndim == 1: # Binary classification, single probability
        y_probs = np.stack([1 - y_probs, y_probs], axis=1)

    num_classes = y_probs.shape[1]
    y_true_one_hot = np.eye(num_classes)[y_true]

    ece = 0
    for i in range(num_classes):
        prob_true = y_true_one_hot[:, i]
        prob_pred = y_probs[:, i]

        bin_boundaries = np.linspace(0, 1, n_bins + 1)
        bin_lowers = bin_boundaries[:-1]
        bin_uppers = bin_boundaries[1:]

        for bin_lower, bin_upper in zip(bin_lowers, bin_uppers):
            # Filter samples falling into the current bin
            in_bin = (prob_pred > bin_lower) & (prob_pred <= bin_upper)
            bin_size = np.sum(in_bin)

            if bin_size > 0:
                bin_accuracy = np.mean(prob_true[in_bin])
                bin_confidence = np.mean(prob_pred[in_bin])
                ece += np.abs(bin_accuracy - bin_confidence) * (bin_size / len(y_true))

    return ece

# Function to find the teacher experiment directory
def find_teacher_exp_dir(lang: str, root_dir: Path):
    lang_root = root_dir / lang
    if not lang_root.exists():
        return None
    candidates = [d for d in lang_root.iterdir() if d.is_dir() and d.name.startswith("teacher_")]
    if not candidates:
        candidates = [d for d in lang_root.iterdir() if d.is_dir() and (d / "best_model").exists()]
    if not candidates:
        return None
    return sorted(candidates)[-1]


# -- main loop over languages --
langs = sorted([d.name for d in DATA_SPLITS_ROOT.iterdir() if d.is_dir()])
if not langs:
    print(f"No language split directories found under {DATA_SPLITS_ROOT}.")

results = {}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for lang in langs:
    print("\n" + "="*80)
    print(f"[LANG] {lang}")
    lang_dir = DATA_SPLITS_ROOT / lang

    teacher_exp_dir = find_teacher_exp_dir(lang, TRAIN_PIPELINE_OUTPUT_ROOT)
    if not teacher_exp_dir:
        print(f"[SKIP] Could not find teacher experiment dir for {lang}. Skipping ECE/Brier calculation.")
        continue

    model_dir = teacher_exp_dir / "best_model"
    if not model_dir.exists():
        print(f"[SKIP] Best model directory not found for {lang}. Skipping ECE/Brier calculation.")
        continue

    try:
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        teacher_tokenizer = AutoTokenizer.from_pretrained(str(model_dir))
        teacher_model = AutoModelForSequenceClassification.from_pretrained(str(model_dir))
        teacher_model.to(DEVICE)
        teacher_model.eval()
        print(f"[OK] Loaded model and tokenizer for {lang}")
    except Exception as e:
        print(f"[ERR] Could not load model/tokenizer for {lang}: {e}. Skipping.")
        continue

    label_map, inv_label_map, metadata = read_label_map(lang_dir)
    num_labels = len(label_map)
    print(f"[INFO] num_labels = {num_labels}")

    test_df = load_dataframe(lang_dir, "test")
    print(f"[INFO] Loaded test data: {len(test_df)} rows")

    # Prepare test entries and dataset
    test_entries = []
    CHUNK_MAX_LEN = int(metadata.get("chunk_max_len", 256))
    CHUNK_STRIDE = int(metadata.get("chunk_stride", 64))

    for idx, row in test_df.iterrows():
        file_path = str(row.get("file_path", "")) if 'file_path' in row else None
        text = read_text_from_row(row)
        chunks = chunk_text_with_tokenizer(teacher_tokenizer, text, max_len=CHUNK_MAX_LEN, stride=CHUNK_STRIDE)
        if 'label_id' in row and not pd.isna(row['label_id']):
            label_id = int(row['label_id'])
        else:
            label_id = int(label_map.get(str(row['label']), -1)) # Use -1 for unknown labels
        if label_id == -1:
             print(f"[WARN] Skipping row {idx} in {lang}/test.csv due to unknown label: {row.get('label', 'N/A')}")
             continue
        test_entries.append({
            "file_path": str(file_path) if file_path else f"{lang}_generated_{idx}",
            "label_id": label_id,
            "chunks": chunks
        })

    if not test_entries:
        print(f"[WARN] No valid test entries found for {lang}. Skipping ECE/Brier calculation.")
        continue

    test_dataset = ChunkDataset(test_entries)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn) # Use a larger batch size for inference

    # Collect logits and true labels for file-level aggregation
    file_logits = {fe['file_path']: [] for fe in test_entries}
    file_true_labels = {fe['file_path']: fe['label_id'] for fe in test_entries}

    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f"{lang} test inference"):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            outputs = teacher_model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits.detach().cpu().numpy()

            for i, fp in enumerate(batch['file_paths']):
                 file_logits[fp].append(logits[i])

    # Aggregate chunk logits to file level (mean)
    aggregated_file_probs = {}
    y_true_list = []
    y_prob_list = [] # Store probabilities for ECE/Brier

    for fp, logits_list in file_logits.items():
        true_label_id = file_true_labels[fp]
        if logits_list:
            # Stack logits and take mean
            mean_logits = np.mean(np.vstack(logits_list), axis=0)
            # Apply softmax to get probabilities
            exp_logits = np.exp(mean_logits - np.max(mean_logits)) # for numerical stability
            probs = exp_logits / np.sum(exp_logits)
            aggregated_file_probs[fp] = probs
            y_true_list.append(true_label_id)
            y_prob_list.append(probs)
        else:
            # Handle files with no chunks if necessary (e.g., assign uniform probability)
            uniform_probs = np.ones(num_labels) / num_labels
            aggregated_file_probs[fp] = uniform_probs
            y_true_list.append(true_label_id)
            y_prob_list.append(uniform_probs)


    y_true_np = np.array(y_true_list)
    y_probs_np = np.array(y_prob_list)

    # Calculate ECE
    ece = expected_calibration_error(y_true_np, y_probs_np, n_bins=10)

    # Calculate Brier Score (multi-class)
    # Brier score is typically calculated per class for multi-class problems
    brier_scores_per_class = []
    for i in range(num_labels):
        y_true_class = (y_true_np == i).astype(int)
        y_prob_class = y_probs_np[:, i]
        brier_scores_per_class.append(brier_score_loss(y_true_class, y_prob_class))

    mean_brier_score = np.mean(brier_scores_per_class)

    print(f"[RESULTS] {lang} - ECE: {ece:.4f}, Mean Brier Score: {mean_brier_score:.4f}")

    results[lang] = {
        "ece": ece,
        "mean_brier_score": mean_brier_score,
        "brier_scores_per_class": {inv_label_map.get(i, str(i)): score for i, score in enumerate(brier_scores_per_class)}
    }

# Save results
output_results_path = Path("calibration_metrics_test_set.json")
json.dump(results, open(output_results_path, "w", encoding="utf8"), indent=2, ensure_ascii=False)
print("\nCalibration metrics calculated and saved to:", output_results_path.resolve())

NameError: name 'DATA_SPLITS_ROOT' is not defined

In [2]:
import json
from pathlib import Path

output_results_path = Path("calibration_metrics_test_set.json")

if output_results_path.exists():
    with open(output_results_path, 'r', encoding='utf8') as f:
        results = json.load(f)
    print(json.dumps(results, indent=2, ensure_ascii=False))
else:
    print(f"File not found: {output_results_path}")

File not found: calibration_metrics_test_set.json


Based on the output of the training pipeline and sanity check, we can infer the following about ECE (Expected Calibration Error) and its potential "barrier" in the context of this model training for English, Hindi, and Marathi:

While the provided code doesn't explicitly calculate or mention ECE, it's a common metric used to evaluate the calibration of classification models, especially in scenarios involving confidence scores (like those produced by the `outputs.logits` in the model).

*   **ECE (Expected Calibration Error):** ECE measures how well a model's predicted probabilities match the actual probabilities of correctness. A low ECE indicates that when the model predicts a class with, say, 80% probability, it is indeed correct approximately 80% of the time for samples where it made that prediction. A high ECE suggests miscalibration, where the model is either overconfident (predicting high probabilities when incorrect) or underconfident (predicting low probabilities when correct).

*   **"Barrier" in this context:** The "barrier" likely refers to a practical or theoretical limit on how low the ECE can be reduced for a given task, dataset, and model architecture. Factors contributing to this barrier could include:
    *   **Dataset characteristics:** Noise, class imbalance, ambiguity in the data, or inherent difficulty of the classification task for certain labels can make perfect calibration challenging.
    *   **Model limitations:** The model's capacity, architecture, or the pre-training data might not be ideally suited for perfectly modeling the complex decision boundaries required for optimal calibration on this specific task.
    *   **Training process:** Hyperparameters, optimization strategies, and regularization can all influence calibration. The early stopping based on macro F1 might not directly optimize for ECE, potentially leaving room for calibration improvement even if F1 is high.
    *   **Language-specific challenges:** Different languages might have varying levels of data availability, script complexity, or linguistic nuances that impact how well a multilingual model can be calibrated across all of them simultaneously.

**Observations from the output for English, Hindi, and Marathi:**

The output shows training and validation macro F1 scores and losses for each language. We see that the macro F1 scores vary across languages (e.g., Hindi reached a higher macro F1 than Marathi). While macro F1 is a measure of classification performance, it doesn't directly tell us about calibration. A model with a high F1 can still be poorly calibrated.

The fact that the model's performance (macro F1) plateaued and early stopping was triggered suggests that for each language, the model reached a point where further training under the current configuration didn't lead to significant improvement in macro F1 on the validation set. This plateau could be indicative of hitting a performance "barrier" for that specific metric. Similarly, there might be an underlying ECE "barrier" that is harder to overcome after reaching a certain level of classification accuracy.

To understand the ECE and its barrier more directly, you would need to:
1.  Calculate ECE on the validation and test sets using the model's predicted probabilities.
2.  Analyze how ECE changes during training alongside macro F1 and loss.
3.  Compare ECE across different languages and potentially different model architectures or training strategies to understand the factors contributing to the "barrier."